## Data preprocessing

In [10]:
import json
import numpy as np
import nltk
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from gensim.models import KeyedVectors
from gensim.models import Word2Vec
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import random
import pickle
from sklearn.metrics import precision_recall_fscore_support
from keras.models import load_model

In [3]:
nltk.download('punkt')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
	"""Preprocesses text by lowercasing, tokenizing, removing stopwords and stemming."""
	tokens = word_tokenize(text.lower())
	filtered_tokens = [stemmer.stem(word) for word in tokens if word.isalnum() and word not in stop_words]
	return ' '.join(filtered_tokens)
	
# Load JSON data
def load_data(filepath):
	with open(filepath, 'r') as file:
		data = json.load(file)
	return data


[nltk_data] Downloading package punkt to /Users/chenluyao/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [4]:
train_claims_data = load_data('data/train-claims.json')
evidence_data = load_data('data/evidence.json')
dev_claims_data = load_data('data/dev-claims.json')

In [5]:
# evidence_map = {eid: preprocess_text(text) for eid, text in evidence_data.items()}

# def save_to_json(filepath, data):
#     """ Save a dictionary to a JSON file. """
#     with open(filepath, 'w', encoding='utf-8') as f:
#         json.dump(data, f, ensure_ascii=False, indent=4)

# # Example usage
# save_to_json('data/curated/preprocessed_evidence_map.json', evidence_map)

In [6]:
evidence_map = load_data('data/curated/preprocessed_evidence_map.json')

## Train Test Split

In [65]:
from sklearn.model_selection import train_test_split

claim_ids = []
for claim_id, claim_details in train_claims_data.items():
	claim_ids.append(claim_id)

# split the claims_df into training and test sets
train, test = train_test_split(claim_ids, test_size=0.2)
len(train)

982

In [68]:
train_data_for_dataframe = []
test_data_for_dataframe = []
evidence_keys = list(evidence_map.keys())  # List of all evidence IDs

for claim_id, claim_details in train_claims_data.items():
	claim_text = preprocess_text(claim_details['claim_text'])
	claim_evidences = set(claim_details['evidences'])  # Convert to set for faster checks

	# Add positive examples
	for eid in claim_evidences:
		evidence_text = evidence_map.get(eid, "NULL")  
		if evidence_text != "NULL":
			data = {
				'claim': claim_text,
				'evidence': evidence_text,
				'label': 1  # Label as relevant
			}
			if claim_id in train:
				train_data_for_dataframe.append(data)
			else:
				test_data_for_dataframe.append(data)

	# Add negative examples
	num_neg_samples = min(len(claim_evidences), len(evidence_keys) - len(claim_evidences))  # Limit the number of negative samples
	negative_samples = random.sample([k for k in evidence_keys if k not in claim_evidences], num_neg_samples)
	for eid in negative_samples:
		evidence_text = evidence_map[eid]
		data = {
			'claim': claim_text,
			'evidence': evidence_text,
			'label': 0  # Label as not relevant
		}
		if claim_id in train:
				train_data_for_dataframe.append(data)
		else:
			test_data_for_dataframe.append(data)

train_df = pd.DataFrame(train_data_for_dataframe)
test_df = pd.DataFrame(test_data_for_dataframe)

In [69]:
train_df

,claim,evidence,label
0,scientif evid co2 pollut higher co2 concentr a...,high concentr 100 time atmospher concentr grea...,1
1,scientif evid co2 pollut higher co2 concentr a...,plant grow much 50 percent faster concentr ppm...,1
2,scientif evid co2 pollut higher co2 concentr a...,higher carbon dioxid concentr favour affect pl...,1
3,scientif evid co2 pollut higher co2 concentr a...,serv architectur transport barrier complianc b...,0
4,scientif evid co2 pollut higher co2 concentr a...,later unit state enter war hx convoy began new...,0
...,...,...,...
6573,send oscil microwav antenna insid vacuum elect...,dielectr heat also known electron heat radio f...,1
6574,send oscil microwav antenna insid vacuum elect...,howev 1830 gener elect top poll canterburi vot...,0
6575,send oscil microwav antenna insid vacuum elect...,screen 2013 macworld iworld event san francisc...,0
6576,send oscil microwav antenna insid vacuum elect...,white support law literatur movement,0


In [93]:
def text2seq(train_text, test_text, tokenizer_name):
	tokenizer = Tokenizer()
	tokenizer.fit_on_texts(train_text)
	input_text_index = tokenizer.word_index # return dictionary of wordss {'the':1, 'earth':2, 'is':3}

	with open(tokenizer_name+'.pickle', 'wb') as handle:
		pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

	max_length = max([len(s.split()) for s in train_text])
	print ("max length:", max_length)

	train_sequence = tokenizer.texts_to_sequences(train_text)
	test_sequence = tokenizer.texts_to_sequences(test_text)
	return (train_sequence, input_text_index, test_sequence, max_length)

def to_padding(train_df, test_df):
	# Initialize and fit the tokenizer on claim and evidence separately
	x_claims_seq, x_claims_word_index, y_claims_seq, max_claims_length = text2seq(train_df["claim"].tolist(), test_df["claim"].tolist(), "tokenizer_claims")
	x_sents_seq, x_sents_word_index, y_sents_seq, max_sents_length = text2seq(train_df["evidence"].tolist(), test_df["evidence"].tolist(), "tokenizer_evidence")

	x_claims_data = pad_sequences(x_claims_seq, maxlen=max_claims_length)  #returns array of data
	x_sents_data = pad_sequences(x_sents_seq, maxlen=max_sents_length)
	x_labels = train_df['label'].values

	y_claims_data = pad_sequences(y_claims_seq, maxlen=max_claims_length)
	y_sents_data = pad_sequences(y_sents_seq, maxlen=max_sents_length)
	y_labels = test_df['label'].values

	return (x_claims_data, x_sents_data, x_labels, x_claims_word_index,  x_sents_word_index, y_claims_data, y_sents_data, y_labels)

def create_embedding_matrix(vocab_size, word_vectors, word_index, embedding_dim):
	embedding_matrix = np.zeros((vocab_size, embedding_dim))
	for word, i in word_index.items():
		if word in word_vectors:
			embedding_vector = word_vectors[word]
			if embedding_vector is not None:
				embedding_matrix[i] = embedding_vector
	return embedding_matrix, embedding_dim

In [104]:
x_claim, x_sents, x_labels, x_claims_word_index,  x_sents_word_index, y_claims_data, y_sents_data, y_labels = to_padding(train_df, test_df)
loss = 'binary_crossentropy'

print ("x claim word index ", len(x_claims_word_index))
print ("x sent word index ", len(x_sents_word_index))

vocab_size_claims = len(x_claims_word_index) + 1
vocab_size_evidences = len(x_sents_word_index) + 1


max length: 35
max length: 180
x claim word index  2503
x sent word index  13067


## Creating the Embedding Matrix

In [106]:
# # We need data for training the model
# all_texts = list(set(train_df['claim'].tolist())) + list(set(evidence_map.values()))
# processed_sentences = [sent.split() for sent in all_texts]

# model = Word2Vec(
# 	sentences=processed_sentences,
# 	vector_size=300
# )
# model.save("word2vec.model")
# word_vectors = model.wv
# word_vectors.save("word2vec.wordvectors")

In [107]:
# Load pre-trained word2vec
# word_vectors = KeyedVectors.load_word2vec_format('GoogleNews-vectors-negative300.bin.gz', binary=True)
word_vectors = KeyedVectors.load('word2vec.wordvectors', mmap='r')

In [108]:
embedding_dim = 300  # dimension of word2vec vectors
(embed_matrix_claim, embed_dim_claim) = create_embedding_matrix(vocab_size_claims, word_vectors, x_claims_word_index, embedding_dim)
(embed_matrix_evidence, embed_dim_evidence) = create_embedding_matrix(vocab_size_evidences, word_vectors, x_sents_word_index, embedding_dim)

print ("embed_matrix_claim shape ", embed_matrix_claim.shape)
print ("embed_matrix_evidence shape ", embed_matrix_evidence.shape)

embed_matrix_claim shape  (2504, 300)
embed_matrix_evidence shape  (13068, 300)


## Building the LSTM Model

We will build a simple unidirectional LSTM model to compare claim and evidence embeddings.

In [120]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Dropout, concatenate, Conv1D, GlobalMaxPooling1D, BatchNormalization
from tensorflow.keras import regularizers

# Define the model
def create_model(vocab_size_claims, vocab_size_evidences, maxlen_claims, maxlen_evidences, 
								embed_dim_claim, embed_dim_evidence):
		
	claims_input = Input(shape=(None,), dtype='int32', name='claims')
	embed_claims = Embedding(vocab_size_claims, embed_dim_claim)(claims_input)
	encoded_claims = LSTM(256, return_sequences=True)(embed_claims)
	encoded_claims = LSTM(16)(encoded_claims)
	encoded_claims = BatchNormalization()(encoded_claims)
		
	evidences_input = Input(shape=(None,), dtype='int32', name='evidences')
	embed_evidences = Embedding(vocab_size_evidences, embed_dim_evidence)(evidences_input)
	encoded_evidences = LSTM(256, return_sequences=True)(embed_evidences)
	encoded_evidences= LSTM(64)(encoded_evidences)
	encoded_evidences = BatchNormalization()(encoded_evidences)
		
	concatenate_layers = concatenate([encoded_claims, encoded_evidences], axis=-1)
	concatenate_layers = Dropout(0.5)(concatenate_layers)
	concatenate_layers = Dense(64, activation='relu')(concatenate_layers)
	pred_label = Dense(1, activation='sigmoid')(concatenate_layers)

	model = Model(inputs=[claims_input, evidences_input], outputs=pred_label)
	model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

	return model


max_claims_length = 35
max_sents_length = 180
model = create_model(vocab_size_claims, vocab_size_evidences, max_claims_length, max_sents_length, 
								embed_dim_claim, embed_dim_evidence)
print(model.summary())


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ claims (InputLayer) │ (None, None)      │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ evidences           │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_5         │ (None, None, 300) │    751,200 │ claims[0][0]      │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_6         │ (None, None, 300) │  3,920,400 │ evidences[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_10 (LSTM)      │ (None, None, 256) │    570,368 │ embedding_5[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_12 (LSTM)      │ (None, None, 256) │    570,368 │ embedding_6[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_11 (LSTM)      │ (None, 16)        │     17,472 │ lstm_10[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_13 (LSTM)      │ (None, 64)        │     82,176 │ lstm_12[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16)        │         64 │ lstm_11[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ lstm_13[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 80)        │          0 │ batch_normalizat… │
│ (Concatenate)       │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 80)        │          0 │ concatenate_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 64)        │      5,184 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 1)         │         65 │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,917,553 (22.57 MB)

 Trainable params: 5,917,393 (22.57 MB)

 Non-trainable params: 160 (640.00 B)

None


In [121]:
model.layers[2].set_weights([embed_matrix_claim])
model.layers[2].trainable = False
model.layers[3].set_weights([embed_matrix_evidence])
model.layers[3].trainable = False
print(model.summary())

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ claims (InputLayer) │ (None, None)      │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ evidences           │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_5         │ (None, None, 300) │    751,200 │ claims[0][0]      │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_6         │ (None, None, 300) │  3,920,400 │ evidences[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_10 (LSTM)      │ (None, None, 256) │    570,368 │ embedding_5[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_12 (LSTM)      │ (None, None, 256) │    570,368 │ embedding_6[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_11 (LSTM)      │ (None, 16)        │     17,472 │ lstm_10[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_13 (LSTM)      │ (None, 64)        │     82,176 │ lstm_12[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16)        │         64 │ lstm_11[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ lstm_13[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 80)        │          0 │ batch_normalizat… │
│ (Concatenate)       │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 80)        │          0 │ concatenate_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 64)        │      5,184 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 1)         │         65 │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,917,553 (22.57 MB)

 Trainable params: 1,245,793 (4.75 MB)

 Non-trainable params: 4,671,760 (17.82 MB)

None


#  Training the Model

In [122]:
from keras.callbacks import EarlyStopping, ModelCheckpoint, Callback, CSVLogger
early_stopping = EarlyStopping(monitor='val_loss', patience=2)

csv_logger = CSVLogger('lstm_training_tf.log')
model_path = 'lstm_evidence_retrieval.keras'
history = model.fit({'claims': x_claim, 'evidences': x_sents}, x_labels, 
						epochs=60, batch_size=64, validation_split=0.12, callbacks=[early_stopping, csv_logger,
						ModelCheckpoint(filepath=model_path, monitor='val_loss', save_best_only=True)])	

Epoch 1/60
91/91 ━━━━━━━━━━━━━━━━━━━━ 91s 943ms/step - accuracy: 0.8718 - loss: 0.3094 - val_accuracy: 0.9241 - val_loss: 0.2232
Epoch 2/60
91/91 ━━━━━━━━━━━━━━━━━━━━ 83s 912ms/step - accuracy: 0.9339 - loss: 0.1797 - val_accuracy: 0.9165 - val_loss: 0.1936
Epoch 3/60
91/91 ━━━━━━━━━━━━━━━━━━━━ 90s 987ms/step - accuracy: 0.9544 - loss: 0.1253 - val_accuracy: 0.9430 - val_loss: 0.1861
Epoch 4/60
91/91 ━━━━━━━━━━━━━━━━━━━━ 95s 1s/step - accuracy: 0.9650 - loss: 0.1095 - val_accuracy: 0.9329 - val_loss: 0.1821
Epoch 5/60
91/91 ━━━━━━━━━━━━━━━━━━━━ 87s 944ms/step - accuracy: 0.9659 - loss: 0.0885 - val_accuracy: 0.9342 - val_loss: 0.2208
Epoch 6/60
91/91 ━━━━━━━━━━━━━━━━━━━━ 96s 1s/step - accuracy: 0.9758 - loss: 0.0715 - val_accuracy: 0.9304 - val_loss: 0.2184


## Test the model

In [126]:
model = load_model("lstm_evidence_retrieval.keras")
loss, accuracy = model.evaluate({'claims': y_claims_data, 'evidences': y_sents_data}, y_labels)
print ("test loss ", loss)
print ("test accuracy ", accuracy)

y_pred = (np.asarray(model.predict({'claims': y_claims_data, 'evidences': y_sents_data} , batch_size=16))).round()
print ("score of lstm ", precision_recall_fscore_support(y_labels, y_pred, average='binary'))

53/53 ━━━━━━━━━━━━━━━━━━━━ 15s 264ms/step - accuracy: 0.9397 - loss: 0.1913
test loss  0.19159094989299774
test accuracy  0.9345738291740417
105/105 ━━━━━━━━━━━━━━━━━━━━ 24s 224ms/step
score of lstm  (0.9570707070707071, 0.9099639855942377, 0.932923076923077, None)


## Test the model on Dev Set

In [130]:
data_for_dataframe = []
for claim_id, claim_details in dev_claims_data.items():
	claim_text = preprocess_text(claim_details['claim_text'])
	claim_evidences = set(claim_details['evidences'])  # Convert to set for faster checks

	# Add positive examples
	for eid in claim_evidences:
		evidence_text = evidence_map.get(eid, "NULL")  
		if evidence_text != "NULL":
			data_for_dataframe.append({
				'claim_id': claim_id,
				'claim': claim_text,
				'evidence_id': eid,
				'evidence': evidence_text,
				'label': 1  # Label as relevant
			})

	# Add negative examples
	negative_samples = [key for key in evidence_map.keys() if key not in claim_evidences]
	for eid in negative_samples:
		evidence_text = evidence_map[eid]
		data_for_dataframe.append({
			'claim_id': claim_id,
			'claim': claim_text,
			'evidence_id': eid,
			'evidence': evidence_text,
			'label': 0  # Label as not relevant
		})
	
# Create DataFrame
dev_df = pd.DataFrame(data_for_dataframe)
dev_df 

In [8]:
data_for_dataframe = []
for claim_id, claim_details in dev_claims_data.items():
    claim_text = preprocess_text(claim_details['claim_text'])
    eids = claim_details['evidences']
    data_for_dataframe.append({
			'claim_id': claim_id,
            'claim': claim_text,
            'evidence': eids
        })
    
# Create DataFrame
dev_claims_df = pd.DataFrame(data_for_dataframe)
dev_claims_df 

,claim_id,claim,evidence
0,claim-752,south australia expens electr world,"[evidence-67732, evidence-572512]"
1,claim-375,3 per cent total annual global emiss carbon di...,"[evidence-996421, evidence-1080858, evidence-2..."
2,claim-1266,mean world 1c warmer time,"[evidence-889933, evidence-694262]"
3,claim-871,happen zika may also good model second worri e...,"[evidence-422399, evidence-702226, evidence-28..."
4,claim-2164,greenland lost tini fraction ice mass,"[evidence-52981, evidence-264761, evidence-947..."
...,...,...,...
149,claim-2400,suddenli label co2 pollut disservic ga play en...,"[evidence-409365, evidence-127519, evidence-85..."
150,claim-204,natur orbit driven warm atmospher carbon dioxi...,"[evidence-368192, evidence-261690, evidence-20..."
151,claim-1426,mani world coral reef alreadi barren state con...,"[evidence-1124018, evidence-995813, evidence-1..."
152,claim-698,recent studi led lawrenc livermor nation labor...,[evidence-660755]


In [11]:
with open('tokenizer_claims.pickle', 'rb') as handle:
	claims_tokenizer = pickle.load(handle)

with open('tokenizer_evidence.pickle', 'rb') as handle:
	sents_tokenizer = pickle.load(handle)

max_claims_length = 35
max_sents_length = 180

model = load_model("lstm_evidence_retrieval.keras")

In [15]:
test_claims = claims_tokenizer.texts_to_sequences(dev_claims_df["claim"])
test_sents = sents_tokenizer.texts_to_sequences(evidence_map.values())

test_claims = pad_sequences(test_claims, maxlen=max_claims_length)
test_sents = pad_sequences(test_sents, maxlen=max_sents_length)
print ("test claims ", test_claims.shape)
print ("test sents ", test_sents.shape)

test claims  (154, 35)
test sents  (1208827, 180)


In [ ]:
#TODO: make claim-evidence pairs without crashing the kernel


In [ ]:
test_claims = claims_tokenizer.texts_to_sequences(dev_df["claim"])
test_sents = sents_tokenizer.texts_to_sequences(dev_df["evidence"])

test_claims = pad_sequences(test_claims, maxlen=max_claims_length)  #returns array of data
test_sents = pad_sequences(test_sents, maxlen=max_sents_length)

print ("test claims ", test_claims.shape)
print ("test sents ", test_sents.shape)
	
loss, accuracy = model.evaluate({'claims':test_claims, 'evidences': test_sents}, test_model.test_data["label"])
print ("test loss ", loss)
print ("test accuracy ", accuracy)

batch_size = 32
y_pred = (np.asarray(model.predict({'claims': test_claims, 'evidences': test_sents} , batch_size=batch_size))).round()
print ("score of lstm ", precision_recall_fscore_support(test_model.test_data["label"], y_pred, average="binary")) 